In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Define save path for 1st model (update as per model)
model_save_path = "/content/drive/MyDrive/BanglaFakeReview/NEW/culturax-base-3b"

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes peft huggingface_hub torch==2.6.0 unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import json
from datasets import Dataset
import pandas as pd

data_path = "/content/drive/MyDrive/BanglaFakeReview/NEW/Dataset/balanced_augmented_dataset.json"  # ⬅️ Your dataset path
with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# ✅ Clean data: Ensure 'Review' is str and 'Label' is int
cleaned_data = []
for item in raw_data:
    review = str(item.get("Review", ""))
    label = int(float(item.get("Label", 1)))  # Safe conversion if label is float-like
    cleaned_data.append({"Review": review, "Label": label})

# ✅ Create HF dataset and shuffle
dataset = Dataset.from_list(cleaned_data).shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print("✅ Dataset loaded and split:")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")


✅ Dataset loaded and split:
Train size: 9920
Test size: 2480


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "BanglaLLM/BanglaLLama-3.2-3b-unlop-culturax-base-v0.0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def prompt_format(example):
    prompt = f"এই রিভিউটি কি ভুয়া নাকি আসল? রিভিউ:\n\"{example['Review']}\"\n\nউত্তর (ভুয়া বা আসল):"
    label = " ভুয়া" if example['Label'] == 0 else " আসল"
    example_text = prompt + label

    tokenized = tokenizer(
        example_text,
        padding="max_length",
        truncation=True,
        max_length=256
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = dataset["train"].map(prompt_format)
tokenized_test = dataset["test"].map(prompt_format)

# ✅ Remove unused columns
tokenized_train = tokenized_train.remove_columns(["Review", "Label"])
tokenized_test = tokenized_test.remove_columns(["Review", "Label"])



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

Map:   0%|          | 0/9920 [00:00<?, ? examples/s]

Map:   0%|          | 0/2480 [00:00<?, ? examples/s]

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True
)

tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
    use_rslora=False,
    loftq_config=None
)

model.print_trainable_parameters()


/tmp/ipython-input-3287559938.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.11: Fast Llama patching. Transformers: 4.54.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.25G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

BanglaLLM/BanglaLLama-3.2-3b-unlop-culturax-base-v0.0.3 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.7.11 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=model_save_path,
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=10,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=None
)



/tmp/ipython-input-1901816393.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import os

# ✅ Resume from last checkpoint (if exists)
checkpoint_path = None
if os.path.isdir(model_save_path):
    checkpoints = [d for d in os.listdir(model_save_path) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoint_path = os.path.join(model_save_path, sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1])
        print(f"🔁 Resuming training from checkpoint: {checkpoint_path}")
    else:
        print("📦 No checkpoint found. Starting from scratch.")

# ✅ Train
trainer.train(resume_from_checkpoint=checkpoint_path)

🔁 Resuming training from checkpoint: /content/drive/MyDrive/BanglaFakeReview/NEW/culturax-base-3b/checkpoint-620


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,920 | Num Epochs = 2 | Total steps = 620
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss


TrainOutput(global_step=620, training_loss=0.0, metrics={'train_runtime': 0.0104, 'train_samples_per_second': 1899929.025, 'train_steps_per_second': 59372.782, 'total_flos': 8.664013080625152e+16, 'train_loss': 0.0, 'epoch': 2.0})

In [ ]:
import torch
from tqdm import tqdm

print("📝 Generating predictions using prompts...")

model.eval()
pred_labels = []
true_labels = []
review_texts = []

positive_keywords = ["ইতিবাচক", "positive", "ভালো", "ভাল", "great", "good", "অসাধারণ", "সেরা"]
negative_keywords = ["নেতিবাচক", "negative", "খারাপ", "বাজে", "poor", "bad", "terrible", "worst"]

prompt_template = "এই রিভিউটি কি ভুয়া নাকি আসল? রিভিউ:\n\"{review}\"\n\nউত্তর (ভুয়া বা আসল):"

for example in tqdm(dataset["test"], desc="Prompt-based Prediction"):
    review = example["Review"]
    true_label = example["Label"]

    # 📝 Create the prompt
    prompt = prompt_template.format(review=review)

    # Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate the prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            num_beams=1,
            early_stopping=True
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    # Simple keyword matching to determine label
    if any(word in decoded for word in negative_keywords):
        pred = 0  # Fake
    elif any(word in decoded for word in positive_keywords):
        pred = 1  # Non-Fake
    else:
        pred = 1  # Default to Non-Fake

    pred_labels.append(pred)
    true_labels.append(true_label)
    review_texts.append(review)

print("\n🔍 Prompt-based prediction complete.")



📝 Generating predictions using prompts...


Prompt-based Prediction: 100%|██████████| 2480/2480 [38:01<00:00,  1.09it/s]


🔍 Prompt-based prediction complete.


In [ ]:
import pandas as pd
import os

# Create DataFrame from the lists
df_pred = pd.DataFrame({
    "review": review_texts,
    "true_label": true_labels,
    "predicted_label": pred_labels
})

# Define save directory and path for the unolp culturax model
save_dir = "/content/drive/MyDrive/BanglaFakeReview/model_4_BanglaLLama_UnolpCulturax"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = os.path.join(save_dir, "prompt_based_predictions.csv")

# Save CSV
df_pred.to_csv(save_path, index=False)
print(f"✅ Prompt-based predictions saved at: {save_path}")


✅ Prompt-based predictions saved at: /content/drive/MyDrive/BanglaFakeReview/model_4_BanglaLLama_UnolpCulturax/prompt_based_predictions.csv


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

print("📊 Running Evaluation on Prompt-Based Predictions...")

true_labels = df_pred["true_label"].tolist()
pred_labels = df_pred["predicted_label"].tolist()

accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, pred_labels, average='weighted', zero_division=0
)

print(f"\n🔹 Accuracy: {accuracy:.4f}")
print(f"🔹 Precision: {precision:.4f}")
print(f"🔹 Recall: {recall:.4f}")
print(f"🔹 F1-Score: {f1:.4f}")

print("\n🔎 Detailed Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=["Fake", "Non-Fake"], digits=4, zero_division=0))


📊 Running Evaluation on Prompt-Based Predictions...

🔹 Accuracy: 0.5044
🔹 Precision: 0.5140
🔹 Recall: 0.5044
🔹 F1-Score: 0.3933

🔎 Detailed Classification Report:
              precision    recall  f1-score   support

        Fake     0.5251    0.0759    0.1327      1238
    Non-Fake     0.5028    0.9316    0.6531      1242

    accuracy                         0.5044      2480
   macro avg     0.5140    0.5037    0.3929      2480
weighted avg     0.5140    0.5044    0.3933      2480

